# 🌳 Decision Tree Regressor

## 📌 Objective
Build a Decision Tree Regressor to predict continuous values and understand how variance reduction works.

## 📊 Dataset
California Housing Dataset

## ⚙️ Workflow
1. Load Data  
2. Understand Data  
3. Train Model  
4. Apply Pruning  
5. Evaluate Performance  

In [18]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.datasets import fetch_california_housing

# Evaluation
from sklearn.metrics import mean_squared_error, r2_score

In [19]:
# Load California Housing dataset
data = fetch_california_housing()

# Convert to DataFrame
df = pd.DataFrame(data.data, columns=data.feature_names)
df["target"] = data.target

df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 📊 Data Understanding

- Dataset contains housing data  
- Goal: Predict house prices  

Target variable:
- target → house price  

We check:
- data types  
- statistics  
- missing values  

In [20]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
 8   target      20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB


MedInc        0
HouseAge      0
AveRooms      0
AveBedrms     0
Population    0
AveOccup      0
Latitude      0
Longitude     0
target        0
dtype: int64

## 🎯 Feature Selection

All features are numerical, so no encoding required

In [21]:
X = df.drop("target", axis=1)
y = df["target"]

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 🌳 Decision Tree Regressor (Without Pruning)

This model will try to fit data as closely as possible

⚠️ Can easily overfit

In [23]:
model = DecisionTreeRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [24]:
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MSE: 0.495235205629094
R2 Score: 0.622075845135081


## ✂️ Pre-Pruning

Control tree complexity using:

- max_depth  
- min_samples_split  

In [25]:
model_pre = DecisionTreeRegressor(
    max_depth=5,
    min_samples_split=10,
    random_state=42
)

model_pre.fit(X_train, y_train)

y_pred_pre = model_pre.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred_pre))
print("R2 Score:", r2_score(y_test, y_pred_pre))

MSE: 0.5245146178314735
R2 Score: 0.5997321244428706


## ✂️ Post-Pruning

Using ccp_alpha to simplify the tree

In [36]:
path = model.cost_complexity_pruning_path(X_train, y_train)
# Reduce number of models for faster execution
ccp_alphas = path.ccp_alphas[:30]

In [37]:
len(ccp_alphas)

30

In [38]:
models = []

for alpha in ccp_alphas:
    clf = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
    clf.fit(X_train, y_train)
    models.append((clf, alpha))

In [39]:
best_r2 = -1
best_alpha = 0

for clf, alpha in models:
    r2 = clf.score(X_test, y_test)
    
    if r2 > best_r2:
        best_r2 = r2
        best_alpha = alpha

print("Best Alpha:", best_alpha)
print("Best R2:", best_r2)  

Best Alpha: 0.0
Best R2: 0.622075845135081


In [40]:
best_model = DecisionTreeRegressor(
    ccp_alpha=best_alpha,
    random_state=42
)

best_model.fit(X_train, y_train)

print("Final R2:", best_model.score(X_test, y_test))

Final R2: 0.622075845135081


## 📊 Observations

- Decision Tree Regressor overfits easily  
- Pre-pruning reduces complexity  
- Post-pruning improves generalization  

## ✅ Conclusion

- Decision Trees work for regression using variance reduction  
- Pruning is essential to control overfitting  
- Proper tuning improves model performance  